# Build a neural TPP

A neural TPP replaces hand-designed intensity functions with a network that
reads the event history. This notebook assembles one by reimplementing the
library's piecewise-constant intensity model on top of a shared transformer
encoder.

The base class `TransformerParametrizedTPP` provides the encoder, a per-batch
parameter cache, and the causal lookup that maps every query time back to the
history available at that moment. Your subclass fills in **four hooks** — that
is the whole contract:

1. `_get_params` — run the encoder and project its context to head parameters.
2. `_intensities_at_queries` — raw intensity $\lambda(t)$ at arbitrary query times.
3. `_interval_ci_from_params` — the closed-form integral $\int \lambda$ over one interval.
4. `_sequence_intensities` — $\lambda$ at every event position, in a single pass.

Here the head predicts, per type, the `K` positive heights of a **step
function** of the time elapsed since the last event. We fit it to data simulated
from a Hawkes process, where the history matters, and compare it with a Poisson
process that ignores the history.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import torchpp as tpp
from torchpp import TransformerParametrizedTPP
from torchpp.modules.intensity import ConstantBaselineParams, ExpKernelParams
from torchpp.modules.nn import TransformerConfig
from torchpp.training import default_fit_config
from torchpp.utils import format_metrics
from torchpp.utils.math import PiecewiseConstant

## Define the model

The step function lives on the interval after each event: the intensity is a
function of the *relative* time `t - t_last`. `knot_locs` are fixed bin edges
on `[0, bin_t_max]`; the transformer predicts the `K` positive bin heights per
type from the history. `PiecewiseConstant` supplies the interpolation and the
exact integral over a bin range.

Read the four hooks in order — each is only a few lines, and together they are
the entire model:

In [ ]:
class MyPiecewiseConstantTPP(TransformerParametrizedTPP):
    """Piecewise-constant intensity predicted per interval by a transformer.

    In the interval after each event the intensity is a step function of the
    elapsed (relative) time ``t - t_last``: ``λ_d(t) = h_{d,k}`` on bin ``k``.
    The transformer predicts the K positive bin heights ``h`` per type from the
    history; ``knot_locs`` are the fixed bin edges on ``[0, bin_t_max]``.
    """

    def __init__(self, config, num_bins=5, bin_t_max=5.0, **kwargs):
        super().__init__(config, **kwargs)
        self.num_bins = num_bins
        # Head: context embedding -> D*K raw values, softplus'd to positive heights.
        self.head = nn.Linear(config.n_embd, config.num_event_types * num_bins)
        # Fixed uniform bin edges; the first must sit at 0 (start of the support).
        self.register_buffer("knot_locs", torch.linspace(0.0, bin_t_max, num_bins))

    # --- hook 1: project the encoder context to head parameters (B, L, D, K) ---
    def _get_params(self, ts):
        h = self._encode(ts)  # (B, L, n_embd) — the shared encoder
        B, L, _ = h.shape
        raw = self.head(h).view(B, L, self.D, self.num_bins)  # (B, L, D, K)
        return F.softplus(raw)  # positive bin heights

    # --- hook 2: raw intensity at arbitrary query times (B, Q) -> (B, Q, D) ---
    def _intensities_at_queries(self, query_times, ts):
        h = self._get_params_at_queries(query_times, ts)  # (B, Q, D, K), causal gather
        B, Q, D, K = h.shape
        t_start = self._event_time_before_queries(query_times, ts)  # (B, Q)
        dt = (query_times - t_start).unsqueeze(-1).expand(B, Q, D)  # relative time
        flat = PiecewiseConstant.interpolate_flat(self.knot_locs, h.reshape(B * Q * D, K), dt.reshape(B * Q * D))
        return flat.view(B, Q, D)

    # --- hook 3: closed-form per-interval integral ∫λ from the gathered params ---
    def _interval_ci_from_params(self, params, t_lo, t_hi):
        *_, K = params.shape  # params: (..., D, K); bounds: (..., D)
        dt = t_hi - t_lo
        integral = PiecewiseConstant.integrate_flat(self.knot_locs, params.reshape(dt.numel(), K), dt.reshape(-1))
        return integral.view(*dt.shape)

    # --- hook 4: intensity at every event position, in one pass (B, L, D) ---
    def _sequence_intensities(self, ts):
        tp = ts.time_points  # (B, L)
        B, L = tp.shape
        if L == 0:
            return torch.zeros(B, 0, self.D, device=tp.device)
        full = self._full_params_cached(ts)  # (B, L+1, D, K), column 0 = no-history
        K = full.shape[3]
        col = ts.causal_indices  # (B, L): causal column per position
        h = full.gather(1, col.view(B, L, 1, 1).expand(B, L, self.D, K))  # (B, L, D, K)
        tp0 = torch.cat([torch.zeros(B, 1, device=tp.device, dtype=tp.dtype), tp], dim=1)  # (B, L+1)
        dt = (tp - tp0.gather(1, col)).unsqueeze(-1).expand(B, L, self.D)  # relative time
        flat = PiecewiseConstant.interpolate_flat(self.knot_locs, h.reshape(B * L * self.D, K), dt.reshape(-1))
        return flat.view(B, L, self.D)

## Data and a small transformer

We simulate sequences from a two-type Hawkes process with hand-set parameters, so each event raises
the rate of the next ones for a while. We keep the transformer tiny (1 layer, 2 heads, embedding
size 16) so training takes well under a minute on CPU. `TransformerConfig` carries the architecture;
`num_event_types` must match the data, and `seed=0` makes the initial weights reproducible.

In [ ]:
true_model = tpp.ConstBaselineExpKernelHawkesProcess(
    D=2,
    baseline_params=ConstantBaselineParams(mu=torch.tensor([0.3, 0.2])),
    kernel_params=ExpKernelParams(alpha=torch.tensor([[0.5, 0.2], [0.3, 0.4]]),
                                  beta=torch.tensor([[2.0, 1.0], [1.0, 1.5]])),
    token_roles=tpp.TokenRoles(has_origin=True),
)
D, T = 2, 20.0


def simulate(n, seed0):
    seqs = []
    with torch.no_grad():
        for i in range(n):
            s = true_model.sample(None, num_steps=60, rng=seed0 + i).sequence
            keep = s.time_points <= T
            seqs.append(tpp.MVEventData(s.time_points[keep].float(), s.event_types[keep], T_obs=T))
    return seqs


train, val, test = simulate(120, 0), simulate(30, 10_000), simulate(30, 20_000)

config = TransformerConfig(num_event_types=D, n_layer=1, n_head=2, n_embd=16, dropout=0.0)
model = MyPiecewiseConstantTPP(config, num_bins=5, bin_t_max=3.0, seed=0)

## Fit

The neural machinery sits behind the same `model.loss` interface, so `fit` trains it like any other
model. We also fit a Poisson process as a baseline that ignores the history.

In [ ]:
fit_config = default_fit_config(num_steps=300, learning_rate=0.01, eval_freq=100, batch_size=32)
result = tpp.fit(model, train, val, fit_config)
model = result.model
print("validation log-likelihood per event:", [round(v, 3) for v in result.val_ll])

poisson = tpp.fit(tpp.PoissonProcess(D=D), train, val,
                  default_fit_config(num_steps=300, learning_rate=0.05, eval_freq=100)).model

## Evaluate

Both models score through the same `evaluate_tpp` entry point. The neural model uses the history and
should beat the Poisson process. This is a small network trained briefly, so we are showing the
workflow, not a tuned result.

`ll_per_event` is the full-sequence log-likelihood per scored event (the training objective). It also contains the probability of no further event between the last event and the end of the window, so it is a bit lower than `joint_ll`, which splits exactly into `marginal_time_ll` + `cond_type_ll`.

In [ ]:
for name, m in [("neural piecewise-constant", model), ("Poisson", poisson)]:
    metrics = tpp.evaluate_tpp(m, test, compute_marginals=False)
    print(f"{name:26s} ll_per_event = {metrics['ll_per_event']:.3f}   "
          f"time LL = {metrics['marginal_time_ll']:.3f}   type LL = {metrics['cond_type_ll']:.3f}")